# Clean Task2 Benign/Infiltration discrimination follow-up

This is **one exploratory study with two predeclared clean schedules**, not a Task1 retrain or a poisoning experiment. The earlier acquisition run increased Infiltration recall but caused many Benign false alarms. The later prediction-prior correction lowered Benign FPR to 0.9720% on fixed test, but Infiltration recall collapsed to 0.1054% and new-class accuracy to 4.5155%. Thus eight-class detection remains unsolved. Historical test results have already been inspected; do not tune this study from test outcomes.

The input audit found no feature-order or label-array alignment defect in a deterministic 40,000-row **training-only** Benign/Infiltration sample. It did find 23 groups of exact cleaned-feature vectors with conflicting labels within that sample. Such ambiguity is observed, but its contribution to model errors is unknown. The former replay stored 5,000 distinct Benign rows among 25,000 old rows. This study selects 100,000 distinct Benign and 10,000 from each old attack class (140,000 training-only replay rows), transforms them once with the frozen Task1 continual preprocessor in bounded 4,096-row batches, and never uses validation/test rows for replay.

**Locked before training:** same Task1 best, seeded expanded eight-class weights, architecture, frozen preprocessing, replay, seed 42, AdamW LR 0.0003, batch 256, 211,728 draws/epoch (828 steps), max 12 epochs, patience four, and full-validation macro-F1 checkpoint selection. Schedule A allocates 70% of draws to old classes in their original old-training proportions, 20% to Infiltration, 5% Webattack, 5% Portscan. Schedule B allocates 75% old, 15% Infiltration, 5% Webattack, 5% Portscan. The schedule is the only intended A/B training difference; comparison with historical runs is not an isolated ablation.

**Validation-only acceptance:** Benign FPR ≤5%, old accuracy ≥90%, and recall ≥50% **for each** of Infiltration, Webattack and Portscan. Choose highest full-validation macro-F1 among eligible arms; ties use lower Benign FPR, higher Infiltration recall, then arm name. If neither qualifies, save the full tradeoff and do **not** open test. Otherwise test the validation-selected checkpoint exactly once. Report per-class precision/recall/F1, macro-F1, Benign FPR, old/new accuracy and forgetting. A high aggregate accuracy cannot conceal a failed class.

Attach the complete prepared-array folder and the verified full Task1 foundation folder. Enable Kaggle GPU and Internet. Keep replay IDs, checkpoints and metrics private outside Git. If a session stops, attach the complete saved output privately and set `RESUME_RUN_DIR`; resume requires the same runtime/device identity.


In [ ]:
from pathlib import Path
import subprocess, sys
SOURCE_SHA = '6ffb310a0b866986bd7063d5b3a485f0278dd39d'
REPO = Path('/kaggle/working/poison-resilient-continual-nids')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', 'feature/unified-novelty'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() == SOURCE_SHA
print('Verified source:', SOURCE_SHA)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable Kaggle GPU and restart'
PREPARED_DIR = Path('/kaggle/input/notebooks/dvdhere/preparation/review2_large_prepare_20261002T223303_224539Z')
TASK1_RUN_DIR = Path('/kaggle/input/notebooks/dvdhere/full-run-task1/full_task1_20261003T074354_763068Z')
OUTPUT_DIR = Path('/kaggle/working/task2_discrimination_study')
RESUME_RUN_DIR = None  # Or a complete prior output folder, including latest.pt files.
MAX_EPOCHS_THIS_CALL = None  # Optional epoch-boundary pause, e.g. 1.
assert PREPARED_DIR.is_dir() and TASK1_RUN_DIR.is_dir(), 'Attach both complete private inputs'
print('GPU:', torch.cuda.get_device_name(0), 'Prepared:', PREPARED_DIR, 'Task1:', TASK1_RUN_DIR)


In [ ]:
# Small synthetic checks only; no real research training here.
subprocess.run([sys.executable, '-m', 'pytest', '-q',
    'tests/test_task2_discrimination.py', 'tests/test_task2_replay.py',
    'tests/test_full_disk_training.py'], cwd=REPO, check=True)


In [ ]:
command = [sys.executable, 'scripts/run_task2_discrimination.py',
           '--prepared-dir', str(PREPARED_DIR), '--task1-run-dir', str(TASK1_RUN_DIR)]
command += (['--resume-run-dir', str(RESUME_RUN_DIR)] if RESUME_RUN_DIR is not None
            else ['--output-dir', str(OUTPUT_DIR)])
if MAX_EPOCHS_THIS_CALL is not None:
    command += ['--max-epochs-this-call', str(MAX_EPOCHS_THIS_CALL)]
print('Running locked clean study:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)
print('Read protocol.json, both arm histories/validation metrics, selection_lock.json and findings.json. Test metrics exist only if validation accepted an arm.')
